# Curricula: build

Second half of the CV build: joins the mappings to the preprocessed CVs and exports the dataset.
Run it after `1_translate_skills.ipynb` and `2_skill_extract.ipynb`, which read
`mapping/preprocessed_cvs.json` and write one file each, keyed on `id_c`:

| notebook | output |
|---|---|
| `1_translate_skills.ipynb` | `mapping/translated_skills.csv` |
| `2_skill_extract.ipynb` | `mapping/{KB}_skill_map.csv` (+ `_detailed`), one per KB in `KBS` |

`KBS` lists the KBs joined to the CVs, one skill column each (`esco_skills`, `onet_skills`); each
must have been mapped by `2_skill_extract.ipynb`. The language generalization and the
job-category aggregation use ESCO only.

The notebook splits the ESCO languages out of `esco_skills` (each generalized to its language
node), merges them with the declared ones into `esco_languages`, aggregates the job category into
`job_category_agg` (shared with the job offers), and exports `curricula_database.json` and the
12-category subset `datasets/findhr/source/filtered_cvs_dt.json`.

In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
import sys; sys.path.append("../../../")
import pandas as pd

from ast import literal_eval
from pathlib import Path
from typing import Optional, List

from kb_graph import KBGraph

DATA_DIR = Path("../../../datasets/findhr")
# KBs joined to the CVs, each mapped by 2_skill_extract.ipynb.
KBS = ["ESCO", "ONET"]

In [ ]:
# One graph per KB: the language split runs on the graph that produced each skill column.
GRAPH_CACHE = {"ESCO": "occSkill_graph.pkl", "ONET": "onet_occSkill_graph.pkl"}
graphs = {kb: KBGraph.load(Path(f"../../../cache/{GRAPH_CACHE[kb]}")) for kb in KBS}
esco_g = graphs["ESCO"]

In [ ]:
def filter_lang(list_skills: List[str], g: KBGraph = esco_g):
    """Split languages out of a skill list: returns (non-language skills, generalized languages).
    """
    langs = {i for i in list_skills if g.is_language(label=i)}
    return sorted(set(list_skills) - langs), sorted(g.generalize_lang(label=i) for i in langs)

In [ ]:
manual_fix = {
    "Business and administration professionals": "Business and administration",
    "General and keyboard clerks": "Clerical support",
    "Science and engineering professionals": "Science and engineering",
    "Information and communications technicians": "ICT",
    "Sales workers": "Sales and customer service",
    "Legal, social and cultural professionals": "Legal, social and cultural",
    "Labourers in mining, construction, manufacturing and transport": "Elementary and manual labour",
    "Health professionals": "Health and care",
    "Protective services workers": "Personal and protective services",
    "Production and specialised services managers": "Management (production and services)",
    "Personal service workers": "Personal and protective services",
    "Teaching professionals": "Education",
    "Food processing, wood working, garment and other craft and related trades workers": "Skilled trades",
    "Agricultural, forestry and fishery labourers": "Elementary and manual labour",
    "Cleaners and helpers": "Elementary and manual labour",
    "Hospitality, retail and other services managers": "Management (production and services)",
    "Commissioned armed forces officers": "Armed forces",
    "Food preparation assistants": "Elementary and manual labour",
    "Handicraft and printing workers": "Skilled trades",
    "Personal care workers": "Health and care",
}

In [ ]:
# JSON gives lists; the asserts below compare frozensets.
cvs_dt = pd.read_json("mapping/stage1_cvs.json")
for col in ["skills", "languages"]:
    cvs_dt[col] = cvs_dt[col].apply(frozenset)
print(f"{len(cvs_dt)} cvs from mapping/stage1_cvs.json")
cvs_dt.head(10)

In [ ]:
translated = pd.read_csv("mapping/translated_skills.csv",
                        converters={"raw_skills": literal_eval, "eng_skills": literal_eval})

# One mapping file per KB, keyed on id_c; its `esco_skills` column is renamed to the KB's column.
# `mapped_skills` is the translated input the mapping was computed on.
SKILL_COLS = {kb: f"{kb.lower()}_skills" for kb in KBS}
skill_maps = {}
for kb, col in SKILL_COLS.items():
    path = Path(f"mapping/{kb}_skill_map.csv")
    assert path.exists(), f'{path} is missing -- run 2_skill_extract.ipynb with KB = "{kb}"'
    skill_maps[kb] = (pd.read_csv(path, converters={"esco_skills": literal_eval, "raw_skills": literal_eval})
                      .rename(columns={"raw_skills": "mapped_skills", "esco_skills": col}))
    print(f"{kb}: {len(skill_maps[kb])} rows -> {col}")

In [ ]:
skill_maps["ESCO"].head(10)

In [ ]:
skill_maps["ONET"].head(10)

In [ ]:
cvs_dt = cvs_dt.merge(translated[["id_c", "raw_skills", "eng_skills"]], on="id_c", how="left")

# id_c is positional: check that the mappings still match the current CVs.
# The translation must start from the CV's own skills.
assert (cvs_dt["raw_skills"].apply(frozenset) == cvs_dt["skills"]).all(), \
    "translated_skills.csv is stale: delete it and re-run 1_translate_skills.ipynb"

# Each KB mapping must start from the current translation.
for kb, col in SKILL_COLS.items():
    cvs_dt = cvs_dt.merge(skill_maps[kb][["id_c", "mapped_skills", col]], on="id_c", how="left")
    cvs_dt[col] = cvs_dt[col].apply(lambda x: x if isinstance(x, list) else [])
    assert (cvs_dt["mapped_skills"].apply(frozenset) == cvs_dt["eng_skills"].apply(frozenset)).all(), \
        f'{kb}_skill_map.csv is stale: run the prune cell in 1_translate_skills.ipynb, ' \
        f'then 2_skill_extract.ipynb with KB = "{kb}"'
    cvs_dt = cvs_dt.drop(columns=["mapped_skills"])

# From here on `raw_skills` holds the English skills.
cvs_dt["raw_skills"] = cvs_dt["eng_skills"].apply(frozenset)
cvs_dt = cvs_dt.drop(columns=["eng_skills"])

In [ ]:
# Split the languages off every KB column; keep only ESCO's, generalized and merged with the
# declared ones into `esco_languages`.
for kb, col in SKILL_COLS.items():
    split = cvs_dt[col].apply(lambda x: pd.Series(filter_lang(x, graphs[kb])))
    cvs_dt[col] = split[0].apply(frozenset)
    if kb == "ESCO":
        cvs_dt["esco_languages"] = (cvs_dt["languages"].apply(list) + split[1]).apply(frozenset)

In [ ]:
# No KB column may keep a language: ESCO's went to `esco_languages`, the others were dropped.
for kb, col in SKILL_COLS.items():
    assert all(not graphs[kb].is_language(label=i) for i in cvs_dt[col].explode().drop_duplicates().dropna())
assert all(esco_g.is_language(label=i) for i in cvs_dt.esco_languages.explode().drop_duplicates().dropna())

In [ ]:
cvs_dt["job_category_agg"] = cvs_dt["job_category"].apply(lambda x : manual_fix.get(x,x))
cvs_dt.head(10)

In [ ]:
# Rename output columns, keep only the exported ones, and export JSON.
cvs_dt = (
    cvs_dt
    .rename(columns={
        "years_professional_experience": "years_experience", "job_category":"esco_job_category"})
    [["id_c", "job_category_agg", "esco_job_category", "years_experience", "edu_eqf",
      "languages", "esco_languages", "raw_skills", *SKILL_COLS.values()]]
)
cvs_dt.head(10)

In [ ]:
assert not cvs_dt.duplicated().any()
assert not cvs_dt.isna().any().any()

In [ ]:
for col in ["languages", "esco_languages", "raw_skills", *SKILL_COLS.values()]:
    cvs_dt[col] = cvs_dt[col].apply(sorted)

cvs_dt.to_json("curricula_database.json", orient="records", indent=2)

## Subset of the curricula

In [ ]:
subset = ['Business and administration', 'Clerical support', 'Science and engineering','ICT', 'Sales and customer service',
          'Legal, social and cultural', 'Elementary and manual labour', 'Personal and protective services', 'Health and care',
          'Management (production and services)', 'Skilled trades', 'Education']

cvs_dt = pd.read_json("curricula_database.json")
filtered_cvs_dt = cvs_dt[cvs_dt.job_category_agg.isin(subset)]

(DATA_DIR / "source").mkdir(parents=True, exist_ok=True)
filtered_cvs_dt.to_json(DATA_DIR / "source/filtered_cvs_dt.json", orient="records", indent=2)
print(f"{len(filtered_cvs_dt)} / {len(cvs_dt)} cvs -> {DATA_DIR / 'source/filtered_cvs_dt.json'}")

In [ ]:
filtered_cvs_dt.job_category_agg.value_counts().to_dict()